In [54]:
# Import libraries

import pandas as pd, sqlalchemy as sqla, decouple
from sqlalchemy import Integer, String, Numeric
from IPython.display import display

clone_engine = sqla.create_engine(decouple.config("MIS_DB"))
dashboard_engine = sqla.create_engine(decouple.config("MIS_DASHBOARD"))
local_engine = sqla.create_engine(decouple.config("MIS_DB_LOCAL"))
mis_file_path = decouple.config("MIS_FILE_PATH")

In [55]:
# Import RTV data from Excel

rtv_data = pd.read_excel(rf"{mis_file_path}\Report Templates\Daily Sales\RTV Data.xlsx")
rtv_data = rtv_data.rename(columns={"product_class":"class", "product_type":"type"})
display(rtv_data.columns)

Index(['date', 'year', 'month', 'num', 'rr_num', 'fob', 'account_name',
       'account_chain', 'lead_name', 'sales_group', 'bpc_region',
       'account_channel', 'channel_class', 'business_unit', 'account_type',
       'ship_add_1', 'ship_add_2', 'item', 'product_code', 'product_name',
       'brand', 'packaging', 'volume', 'class', 'usage', 'type',
       'product_category', 'qty', 'amount', 'net_amount', 'Remarks',
       'Promo/Regular'],
      dtype='object')

In [56]:
# Select columns from raw rtv_data and filter rows

clean_rtv_data = rtv_data[['year', 'month', 'sales_group', 'account_name', 'account_chain', 'account_type', 'lead_name', 'product_code', 'product_name', 'class', 'brand', 'type', 'net_amount',]]
clean_rtv_data["net_amount"] = clean_rtv_data["net_amount"].abs()

print("clean_rtv_data net_amount: ", clean_rtv_data["net_amount"].sum())

# included_months = ["January", "February", "March", "April", "May", "June"]

rtv_data_2026 = clean_rtv_data[clean_rtv_data["year"] == 2026]
# clean_rtv_data = clean_rtv_data[clean_rtv_data["month"].isin(included_months)]
# display(rtv_data_2026.sum())
print("distinct years: ", clean_rtv_data["year"].unique())
print("distinct months: ", clean_rtv_data["month"].unique())
print("Filtered net_amount: ", clean_rtv_data["net_amount"].sum())

clean_rtv_data net_amount:  29407692.88347524
distinct years:  [2026 2025]
distinct months:  ['July' 'June' 'May' 'April' 'March' 'February' 'January' 'December'
 'November' 'October' 'September' 'August']
Filtered net_amount:  29407692.88347524


C:\Users\bioco\AppData\Local\Temp\ipykernel_14700\1015049001.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_rtv_data["net_amount"] = clean_rtv_data["net_amount"].abs()


In [57]:
# Merge ref.product_details and rtv dataframes 

ref_product_details = pd.read_sql("SELECT * FROM ref.product_details", clone_engine)
final_rtv_data = pd.merge(clean_rtv_data, ref_product_details, how="left", left_on="product_code", right_on="product_code1")
# print(final_rtv_data.columns)

columns_to_drop = [column for column in final_rtv_data.columns if column.endswith('_x')]
final_rtv_data = final_rtv_data.drop(columns=columns_to_drop)
final_rtv_data = final_rtv_data.rename(columns=lambda x: x.replace("_y", ""))
print("Renamed columns: \n", final_rtv_data.columns,"\n")
final_rtv_data = final_rtv_data[['year', 'month', 'sales_group', 'account_name', 'account_chain', 'account_type', 'lead_name', 'product_code', 'main_code', 'product_name', 'class', 'brand', 'type', 'net_amount']]
print("Final columns: \n", final_rtv_data.columns, "\n")

null_rtv = final_rtv_data[ final_rtv_data[['year', 'month', 'sales_group', 'account_name', 'account_chain', 'account_type', 'lead_name', 'product_code', 'main_code', 'product_name', 'class', 'brand', 'type', 'net_amount']].isna().any(axis=1) ]
display(null_rtv)

Renamed columns: 
 Index(['year', 'month', 'sales_group', 'account_name', 'account_chain',
       'account_type', 'lead_name', 'product_code', 'class', 'type',
       'net_amount', 'product_code1', 'product_code2', 'main_code',
       'product_name', 'common_name', 'brand', 'packaging', 'volume',
       'product_class', 'usage', 'product_type', 'product_category',
       'target_type'],
      dtype='object') 

Final columns: 
 Index(['year', 'month', 'sales_group', 'account_name', 'account_chain',
       'account_type', 'lead_name', 'product_code', 'main_code',
       'product_name', 'class', 'brand', 'type', 'net_amount'],
      dtype='object') 



,year,month,sales_group,account_name,account_chain,account_type,lead_name,product_code,main_code,product_name,class,brand,type,net_amount
5284,2026,April,ECOM,Azora Trading and Marketing OPC,E-Commerce,Online,NaN,OND 2025 - B1G5: BACT80,BACT80,OND 2025 - B1G5: iWhite Korea Aqua Moisturizer...,Skin Care,iWhite Korea,Moisturizer,0.0
5285,2026,April,ECOM,Azora Trading and Marketing OPC,E-Commerce,Online,NaN,OND 2025 - B1G4: BFWT,BFWT,OND 2025 - B1G4: iWhite Korea Facial Wash Whit...,Skin Care,iWhite Korea,Facial Wash,0.0
5286,2026,April,ECOM,Azora Trading and Marketing OPC,E-Commerce,Online,NaN,OND 2025 - B1G5: BBLT,BBLT,OND 2025 - B1G5: iWhite Korea BB Holic Light -...,Skin Care,iWhite Korea,BB Cream,0.0
5287,2026,April,ECOM,Azora Trading and Marketing OPC,E-Commerce,Online,NaN,OND 2025 - B1G5: PFWT,PFWT,OND 2025 - B1G5: iWhite Korea Glow-Up Whip Was...,Skin Care,iWhite Korea,Facial Wash,0.0
5288,2026,April,ECOM,Azora Trading and Marketing OPC,E-Commerce,Online,NaN,OND 2025 - B1G5: FCT,FCT,OND 2025 - B1G5: iWhite Korea Facial Cream Whi...,Skin Care,iWhite Korea,Facial Cream,0.0
5289,2026,April,ECOM,Azora Trading and Marketing OPC,E-Commerce,Online,NaN,OND 2025 - B1G5: PACT80,PACT80,OND 2025 - B1G5: iWhite Korea Aqua Moisturizer...,Skin Care,iWhite Korea,Moisturizer,0.0
5290,2026,April,ECOM,Azora Trading and Marketing OPC,E-Commerce,Online,NaN,MXSB,MXSB,Men by iWhite Korea Extra Fresh Soap Bar - 90g...,Skin Care,Men by iWhite Korea,Body Wash,0.0
5291,2026,April,ECOM,Azora Trading and Marketing OPC,E-Commerce,Online,NaN,BACT,BACT,iWhite Korea Aqua Moisturizer Whitening Vita -...,Skin Care,iWhite Korea,Moisturizer,0.0
5292,2026,April,ECOM,Azora Trading and Marketing OPC,E-Commerce,Online,NaN,GACS,GACS,iWhite Korea Aqua Moisturizer Acne+ - 6ml (GACS),Skin Care,iWhite Korea,Moisturizer,0.0
30345,2025,February,ECOM,Tiktok,E-Commerce,Online,NaN,PFWS,PFWS,iWhite Korea Glow-Up Whip Wash - 10ml (PFWS),Skin Care,iWhite Korea,Facial Wash,0.0


In [58]:
# Anti-join

rtv_mtd = final_rtv_data.copy()

try:
    existing = pd.read_sql("SELECT * FROM `dashboard-sales`.rtv", con=dashboard_engine)
    existing = existing.drop(columns=["id"], errors="ignore")

    for df in [rtv_mtd, existing]:
        df["year"] = df["year"].astype("int64")
        df["net_amount"] = pd.to_numeric(df["net_amount"], errors="coerce").round(5)

    key_cols = list(rtv_mtd.columns)
    rtv_mtd["__occ"] = rtv_mtd.groupby(key_cols).cumcount()
    existing["__occ"] = existing.groupby(key_cols).cumcount()

    merged = rtv_mtd.merge(existing, on=key_cols + ["__occ"], how="left", indicator=True)
    new_rows = merged[merged["_merge"] == "left_only"].drop(columns=["_merge", "__occ"])
    rtv_mtd = rtv_mtd.drop(columns=["__occ"])

except Exception:
    new_rows = rtv_mtd

print(f"New rows to insert: {len(new_rows)}")
new_rows.shape

New rows to insert: 31481


(31481, 14)

In [59]:
# Export data to database
final_rtv_data.loc[final_rtv_data["account_chain"] == "E-Commerce", "lead_name"] = "ECOM"
# final_rtv_data = final_rtv_data[final_rtv_data["account_chain"] == "E-Commerce"]
# final_rtv_data.head()
    
final_rtv_data.to_sql(
    name="rtv",
    con=dashboard_engine,
    schema="dashboard-sales",
    if_exists="append",
    index=False,
    chunksize=1000,
    method="multi"
)

31481

In [60]:
# Enforce NOT NULL with no default on every sell_out_staging column
# (dtype= on to_sql only shapes CREATE TABLE; it doesn't alter an existing table)

# alter_sql = """
# ALTER TABLE `dashboard-sales`.`rtv`
#     MODIFY `year`          INT           NOT NULL,
#     MODIFY `month`         VARCHAR(20)   NOT NULL,
#     MODIFY `sales_group`   VARCHAR(50)   NOT NULL,
#     MODIFY `account_name`  VARCHAR(255)  NOT NULL,
#     MODIFY `account_chain` VARCHAR(255)  NOT NULL,
#     MODIFY `account_type`  VARCHAR(100)  NOT NULL,
#     MODIFY `lead_name`     VARCHAR(255)  ,
#     MODIFY `product_code`  VARCHAR(50)   NOT NULL,
#     MODIFY `main_code`     VARCHAR(255)  NOT NULL,
#     MODIFY `product_name`  VARCHAR(255)  NOT NULL,
#     MODIFY `class`         VARCHAR(100)  NOT NULL,
#     MODIFY `brand`         VARCHAR(100)  NOT NULL,
#     MODIFY `type`          VARCHAR(100)  NOT NULL,
#     MODIFY `net_amount`    DECIMAL(15,5) UNSIGNED NOT NULL
# """

# with dashboard_engine.begin() as conn:
#     conn.execute(sqla.text(alter_sql))

In [61]:
rtv_dashboard = pd.read_sql("SELECT * FROM `dashboard-sales`.rtv", dashboard_engine)
rtv_dashboard.head()

,id,year,month,sales_group,account_name,account_chain,account_type,lead_name,product_code,main_code,product_name,class,brand,type,net_amount
0,1,2026,July,S4,RCS Eastern Intertrade Corp.,RCS Eastern Intertrade Corp.,Outright,Vacant - North Luzon,GS 2024 - B2T2: WPS,WPS,GS 2024 - B2T2: iWhite Korea Whitening Pack - ...,Skin Care,iWhite Korea,Pore Care,857.99752
1,2,2026,July,S2,Philippine Seven Corporation,Philippine Seven Corporation,Outright,Rubylie Fabrigas,APWS,APWS,Aera Pore Clear Foam Wash - 10ml (APWS),Skin Care,Aera Essentials,Facial Wash,29.24880
2,3,2026,July,S2,Philippine Seven Corporation,Philippine Seven Corporation,Outright,Rubylie Fabrigas,PFWS,PFWS,iWhite Korea Glow-Up Whip Wash - 10ml (PFWS),Skin Care,iWhite Korea,Facial Wash,18.00120
3,4,2026,July,S2,Philippine Seven Corporation,Philippine Seven Corporation,Outright,Rubylie Fabrigas,MDS,MDS,Men by iWhite Korea 4in1 Deep Facial Wash - 10...,Skin Care,Men by iWhite Korea,Facial Wash,16.49760
4,5,2026,July,S2,Philippine Seven Corporation,Philippine Seven Corporation,Outright,Rubylie Fabrigas,MAS,MAS,Men by iWhite Korea Acne+ Facial Wash - 10ml (...,Skin Care,Men by iWhite Korea,Facial Wash,162.00240
